# Pathview Plus, explained from the beginning

Pathview Plus puts your numbers on top of a pathway diagram.

The first column contains gene identifiers. Every later column is one condition or state.

- One condition → one color across the node.
- Two conditions → two vertical pieces: first column on the left, second column on the right.
- Three conditions → three equal pieces from left to right.

This notebook uses the official `hsa04110` KEGG files bundled with R pathview 1.52.0, so both implementations receive the exact same frozen pathway background.

In [ ]:
from pathlib import Path
import json
import os
import sys

HERE = Path.cwd().resolve()
SEARCH_FOLDERS = (HERE, *HERE.parents)
PROJECT_CANDIDATES = (
    *SEARCH_FOLDERS,
    *(folder / "pygage-pathview-validation" for folder in SEARCH_FOLDERS),
)
ROOT = next(
    (
        folder for folder in PROJECT_CANDIDATES
        if (folder / ".venv").exists() and (folder / "scripts").exists()
    ),
    HERE,
)

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
print("Validation folder:", ROOT)

In [ ]:
import importlib
import shutil
import numpy as np
import polars as pl
from PIL import Image as PILImage
from IPython.display import Image, SVG, display
import pathview
from pathview import SpeciesInfo

print("Pathview Plus distribution: 2.0.2")
print("Runtime __version__:", pathview.__version__)

## Reproducible setup

Current live pathway services can change. For a fair test, this notebook copies the frozen KEGG XML and PNG already installed with Bioconductor pathview.

The small resolver below only prevents an unnecessary live organism-list request. It does not change the pathway data, mapping, color calculation, or renderer.

In [ ]:
CACHE = ROOT / "cache" / "kegg"
CACHE.mkdir(parents=True, exist_ok=True)
r_extdata = ROOT / ".r-library" / "pathview" / "extdata"
for extension in ("xml", "png"):
    shutil.copy2(r_extdata / f"hsa04110.{extension}", CACHE / f"hsa04110.{extension}")

orchestrator = importlib.import_module("pathview.pathview")
original_resolver = orchestrator.kegg_species_code
def frozen_human_resolver(species="hsa"):
    if species in {"hsa", "human", "Homo sapiens"}:
        return SpeciesInfo("hsa", True, None, None, None, None)
    return original_resolver(species)
orchestrator.kegg_species_code = frozen_human_resolver

palette = dict(
    limit={"gene": 2.0, "cpd": 2.0},
    bins={"gene": 11, "cpd": 11},
    both_dirs={"gene": True, "cpd": True},
    low={"gene": "#00FF00", "cpd": "#0000FF"},
    mid={"gene": "#BEBEBE", "cpd": "#BEBEBE"},
    high={"gene": "#FF0000", "cpd": "#FFFF00"},
)

## Part 1 — The basic one-condition pathway

`Classical` is the only value column, so every mapped node gets one full-width color.

In [ ]:
classical = pl.read_csv(
    ROOT / "data" / "classical_hsa04110.csv",
    schema_overrides={"gene_id": pl.String},
)
classical

In [ ]:
one_result = pathview.pathview(
    "04110",
    gene_data=classical,
    species="hsa",
    gene_idtype="ENTREZ",
    kegg_dir=CACHE,
    out_suffix="notebook_classical",
    map_symbol=False,
    new_signature=False,
    plot_col_key=False,
    **palette,
)
one_image = CACHE / "hsa04110.notebook_classical.png"
display(Image(filename=str(one_image), width=800))

## Part 2 — The requested half-and-half pathway

The order is important:

1. `Classical` is first, so it goes on the **left**.
2. `Basal` is second, so it goes on the **right**.

Negative values use green and positive values use red in this controlled palette.

In [ ]:
half = pl.read_csv(
    ROOT / "data" / "half_and_half_hsa04110.csv",
    schema_overrides={"gene_id": pl.String},
)
half

In [ ]:
half_result = pathview.pathview(
    "04110",
    gene_data=half,
    species="hsa",
    gene_idtype="ENTREZ",
    kegg_dir=CACHE,
    out_suffix="notebook_half_half",
    map_symbol=False,
    new_signature=False,
    plot_col_key=False,
    **palette,
)
half_image = CACHE / "hsa04110.notebook_half_half.png"
display(Image(filename=str(half_image), width=800))

In [ ]:
mapped = half_result["plot_data_gene"].filter(pl.col("Classical").is_not_null())
mapped.select("entry_id", "name", "label", "x", "y", "Classical", "Basal")

The table is the strongest first check: the expected values reached the expected pathway coordinates. The image test then checks color direction inside the CDKN2A node.

In [ ]:
half_check = json.loads(
    (ROOT / "results" / "pathview_python" / "validation.json").read_text()
)
pixel_check = next(
    item for item in half_check["checks"]
    if item["name"] == "exact half-and-half pixels on CDKN2A"
)
pl.DataFrame([pixel_check["details"]["pixel_counts"]])

`left_green` and `right_red` are large, while `right_green` and `left_red` are zero. That is a direct pixel-level proof of the requested layout.

In [ ]:
display(Image(
    filename=str(ROOT / "results" / "pathview_python" / "hsa04110.half_half.raw_overlay.png"),
    width=800,
))

## Part 3 — Three conditions

Three value columns make three equal left-to-right bands in the same column order.

In [ ]:
three = pl.read_csv(
    ROOT / "data" / "three_state_hsa04110.csv",
    schema_overrides={"gene_id": pl.String},
)
three_result = pathview.pathview(
    "04110",
    gene_data=three,
    species="hsa",
    gene_idtype="ENTREZ",
    kegg_dir=CACHE,
    out_suffix="notebook_three_state",
    map_symbol=False,
    new_signature=False,
    plot_col_key=False,
    **palette,
)
display(Image(filename=str(CACHE / "hsa04110.notebook_three_state.png"), width=800))

## Part 4 — Output formats

The automated run checked:

- native PNG;
- SVG vector output;
- graph-layout PDF.

Python's native PNG and SVG split multiple states. Its graph/PDF renderer currently uses the first state only, so native PNG is the fair R/Python half-and-half comparison target.

In [ ]:
format_files = [
    ROOT / "results" / "pathview_python" / "hsa04110.half_half.png",
    ROOT / "results" / "pathview_python" / "hsa04110.half_half.svg",
    ROOT / "results" / "pathview_python" / "hsa04110.graph.pdf",
]
pl.DataFrame({
    "file": [path.name for path in format_files],
    "exists": [path.exists() for path in format_files],
    "bytes": [path.stat().st_size for path in format_files],
})

## Part 5 — Paper use cases ready for live KEGG access

The validation package also includes prepared input tables for:

- `hsa04151` PI3K–Akt, one classical condition;
- `hsa04010` MAPK, three conditions;
- `hsa00010` glycolysis, genes plus compounds;
- `ko00910` nitrogen metabolism, KO identifiers.

Run this after live KEGG access is available:

```bash
python scripts/run_pathview_validation.py --live
```

## Pathview Plus conclusion

The reproducible core run passed one-state, two-state, exact left/right pixels, three-state, PNG, SVG, and PDF checks on an official frozen pathway. The full report separates core behavior from live external-service checks and documents every tested public area.